# Support Ops Env — GRPO Training on Colab

**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/spaces/YOUR_HF_USER/support-ops-env) | [GitHub](https://github.com/YOUR_GH_USER/support-ops-env)

Train a SaaS support-ops triage agent using **GRPO** with HF TRL. The agent learns to investigate cases across six apps (Inbox, CRM, Billing, Access, Policy, Comms), apply the right routing, consolidate duplicates, and draft a grounded reply — importing all training logic directly from the `support_ops_env` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://huggingface.co/spaces/YOUR_HF_USER/support-ops-env) (deterministic grader) |
| Training   | This Colab notebook (T4 / A100 GPU) |
| Model      | `Qwen/Qwen2.5-1.5B-Instruct` + LoRA (use `Qwen/Qwen3-0.6B` on T4) |
| Framework  | HF TRL v0.29+ GRPO with vLLM backend |

The training signal is generated **at rollout time** by the agent interacting with the environment — there is no pre-collected dataset.

## 1. Install Dependencies

Installs the `support_ops_env` package (env client, models, tasks, training utils) with the `[train]` extra (TRL + vLLM + LoRA).

In [ ]:
!pip install -q \
    "trl[vllm]>=0.29.0" \
    "vllm>=0.11.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

# Install support_ops_env package (client + models + tasks + training utils)
!pip install -q "openenv-support-ops[train] @ git+https://huggingface.co/spaces/YOUR_HF_USER/support-ops-env"

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets (key icon in the sidebar).

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not found. Set it in Colab Secrets or as env var.")

ENV_URL     = "https://YOUR_HF_USER-support-ops-env.hf.space"
MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"  # use "Qwen/Qwen3-0.6B" on T4
HUB_REPO    = "YOUR_HF_USER/support-ops-agent-qwen25-1.5b"
NUM_EPISODES    = 50
NUM_GENERATIONS = 8
MAX_TURNS       = 15

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Generations : {NUM_GENERATIONS}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment (picks a task), and execute one investigative tool call to confirm round-trip works.

In [ ]:
from support_ops_env import SupportOpsAction, SupportOpsEnv, TASK_IDS

print(f"Connecting to {ENV_URL} ...")

with SupportOpsEnv(base_url=ENV_URL) as env:
    reset_result = env.reset(task_id=TASK_IDS[0])
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Task       : {obs.task_title}")
    print(f"Objective  : {obs.objective[:200]}...\n")

    step = env.step(
        SupportOpsAction(
            assistant_message="Listing inbox to find the primary case.",
            tool_calls=[{"name": "inbox.list_cases", "args": {}}],
            answer=None,
        )
    )
    print(f"--- smoke test step (reward={step.reward:.2f}) ---")
    for tr in step.observation.tool_results[-1:]:
        print(f"{tr.name} ok={tr.ok}\n{str(tr.result)[:400]}")

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

All training logic (system prompt, rollout function, reward functions, helpers) is imported directly from `support_ops_env.train` — the same code used for H100 training. No duplication.

In [ ]:
import csv
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

from support_ops_env import get_training_utils

tu = get_training_utils()
SYSTEM_PROMPT         = tu["SYSTEM_PROMPT"]
rollout_once          = tu["rollout_once"]
reward_total          = tu["reward_total"]
reward_fields         = tu["reward_fields"]
reward_reply          = tu["reward_reply"]
reward_grounding      = tu["reward_grounding"]
plot_rewards          = tu["plot_rewards"]
patch_trl_vllm_compat = tu["patch_trl_vllm_compat"]

patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])
print("...")
print("\nImported: rollout_once, reward_total, reward_fields, reward_reply, reward_grounding.")

## 5. GRPO Training Setup

Configure tokenizer, environment connection, reward logging, and the GRPO trainer — all using functions imported from the package.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

env = SupportOpsEnv(base_url=ENV_URL)
dataset = Dataset.from_dict(
    {"prompt": ["Triage and resolve this support operations case."] * NUM_EPISODES}
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"outputs/support-ops-grpo-{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_totals = []

with open(reward_log_path, "w", newline="") as fh:
    csv.writer(fh).writerow([
        "episode", "total_reward", "field_reward", "reply_reward", "grounding_reward", "timestamp",
    ])

def log_episode(total_r, field_r, reply_r, ground_r):
    episode_counter[0] += 1
    all_totals.append(total_r)
    with open(reward_log_path, "a", newline="") as fh:
        csv.writer(fh).writerow([
            episode_counter[0], total_r, field_r, reply_r, ground_r, datetime.now().isoformat(),
        ])
    last10 = all_totals[-10:]
    logger.info(
        f"Episode {episode_counter[0]}: reward={total_r:+.2f} | "
        f"mean={sum(all_totals)/len(all_totals):+.2f}, mean(10)={sum(last10)/len(last10):+.2f}"
    )

def rollout_func(prompts, trainer):
    results = {k: [] for k in [
        "prompt_ids", "completion_ids", "logprobs",
        "total_reward", "field_reward", "reply_reward", "grounding_reward",
    ]}
    for _ in prompts:
        ep = rollout_once(trainer, env, tokenizer, SYSTEM_PROMPT, MAX_TURNS)
        for k in results:
            results[k].append(ep[k])
        log_episode(
            ep["total_reward"], ep["field_reward"], ep["reply_reward"], ep["grounding_reward"],
        )
    return results

print(f"Training setup complete. Output: {output_dir}")

In [ ]:
grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=1.0,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_fields, reward_reply, reward_grounding],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("GRPOTrainer initialized")

## 6. Train!

Launch GRPO training. Each episode: `reset` → agent investigates via tool calls → agent drafts a reply and submits → deterministic grader scores → GRPO gradient update.

In [ ]:
print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {ENV_URL}")
print()

try:
    trainer.train()
finally:
    if hasattr(env, "close"):
        env.close()

trainer.save_model(str(output_dir))
print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
plot_rewards(reward_log_path, output_dir / "reward_curve.png")

import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to Hugging Face Hub.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")